# 02 SPC and capability: crimp height

X-bar and R charts per machine with Western Electric rules on the simulated crimp height data
(`data/simulated/crimp_height.csv`, built by `python -m src.simulate.crimp`).

This notebook reads only the analysis table. The hidden truth file (`crimp_truth.json`) is not
opened here: the charts have to find the special causes on their own.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))  # make `src` importable when run from notebooks/

In [ ]:
import pandas as pd

from src.plots import xbar_r_chart
from src.spc import chart_table, signal_log

FIGURES = ROOT / "reports" / "figures"
TABLES = ROOT / "reports" / "tables"
TABLES.mkdir(parents=True, exist_ok=True)

data = pd.read_csv(ROOT / "data" / "simulated" / "crimp_height.csv", dtype={"shift": str})
print(
    len(data),
    "parts;",
    data["subgroup"].nunique(),
    "subgroups of",
    data.groupby("subgroup").size().iloc[0],
)
data.head()

## 1. Phase I baseline and limits

Each machine gets its own charts. Limits come from the first 5 production days (30 subgroups
per machine), the Phase I baseline; all 180 subgroups per machine are then checked against
those limits (Phase II). Signals inside the baseline are kept in the log: in a real plant each
one would be investigated before the limits are frozen.

In [ ]:
BASELINE_DAYS = 5
tables, limits = {}, {}
for machine, df in data.groupby("machine"):
    stats_days = (
        df.drop_duplicates("subgroup").sort_values("subgroup")["day"].reset_index(drop=True)
    )
    table, lim = chart_table(df, baseline=stats_days <= BASELINE_DAYS)
    tables[machine], limits[machine] = table, lim

limits_table = (
    pd.DataFrame(limits)
    .T[["xbar_center", "xbar_lcl", "xbar_ucl", "r_center", "r_ucl", "sigma_hat"]]
    .astype(float)
    .round(4)
)
limits_table.to_csv(TABLES / "spc_limits.csv")
limits_table

## 2. Charts

In [ ]:
for machine, table in tables.items():
    xbar_r_chart(
        table,
        limits[machine],
        f"Machine {machine}: X-bar and R charts, crimp height",
        FIGURES / f"spc_xbar_r_machine_{machine}.png",
    )

![Machine A](../reports/figures/spc_xbar_r_machine_A.png)

![Machine B](../reports/figures/spc_xbar_r_machine_B.png)

## 3. Signal log

In [ ]:
log = pd.concat([signal_log(t) for t in tables.values()], ignore_index=True)
log.to_csv(TABLES / "spc_signal_log.csv", index=False)
summary = log.groupby(["machine", "chart"]).agg(
    signals=("subgroup", "size"), first_subgroup=("subgroup", "min"), first_day=("day", "min")
)
summary

In [ ]:
# First signal on each chart after the baseline, and how long signals persist from there.
after = log[~log["baseline"]]
after.groupby(["machine", "chart"]).head(3)

## 4. Stratification by shift

The R chart pools all shifts, so a spread problem on one shift shows up as scattered range
signals. Grouping subgroup ranges by shift makes the pattern visible. sigma = R-bar / d2.

In [ ]:
from src.spc import d2

ranges = pd.concat(tables.values())
by_shift = (
    ranges.groupby(["machine", "shift"])["range"]
    .mean()
    .rename("r_bar")
    .to_frame()
    .assign(sigma_hat=lambda t: t["r_bar"] / d2(5))
)
by_shift["ratio_to_shift_1"] = (
    by_shift["sigma_hat"]
    / by_shift.xs("1", level="shift")["sigma_hat"]
    .reindex(by_shift.index.get_level_values("machine"))
    .to_numpy()
)
r_signals_by_shift = (
    log[log["chart"] == "R"].groupby(["machine", "shift"]).size().rename("r_signals")
)
by_shift = by_shift.join(r_signals_by_shift).fillna({"r_signals": 0}).astype({"r_signals": int})
by_shift.round(4).to_csv(TABLES / "spc_range_by_shift.csv")
by_shift.round(4)